In [ ]:

import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('science_machine')

# Load data
print("Loading data...")
adata = sc.read_h5ad('corrected_filtered_data.h5ad')
de_results = pd.read_csv('de_results_all_drugs.csv')
drugs_metadata = pd.read_csv('drugs.csv')

print(f"Data shape: {adata.shape}")
print(f"DE results shape: {de_results.shape}")
print(f"Drugs metadata shape: {drugs_metadata.shape}")
print(f"\nUnique drugs in DE results: {de_results['drug'].nunique()}")


Loading data...


Data shape: (1204, 7872)
DE results shape: (1322496, 11)
Drugs metadata shape: (173, 25)

Unique drugs in DE results: 168


In [ ]:

# Get summary of DE proteins per drug
de_summary = de_results[de_results['is_de'] == True].groupby('drug').agg({
    'protein': 'count',
    'direction': lambda x: (x == 'up').sum()
}).rename(columns={'protein': 'total_de', 'direction': 'up_count'})
de_summary['down_count'] = de_summary['total_de'] - de_summary['up_count']
de_summary = de_summary.sort_values('total_de', ascending=False)

print("Top 20 drugs by DE proteins:")
print(de_summary.head(20))


Top 20 drugs by DE proteins:
                                total_de  up_count  down_count
drug                                                          
Mobocertinib (succinate)            2907      1008        1899
Clomiphene (citrate)                2737       730        2007
Digitoxin                           2013       147        1866
Olmutinib                           2011       306        1705
Tucidinostat                        1811       744        1067
Abemaciclib (methanesulfonate)      1764       376        1388
Etravirine                          1628       317        1311
Binimetinib                         1223       307         916
Flumatinib                          1217       436         781
Chlorpropamide                      1207       304         903
Remdesivir                          1010       213         797
Enclomiphene (citrate)               945       286         659
Methylene blue (trihydrate)          910       201         709
Cabazitaxel               

In [ ]:

# Examine drugs metadata to understand drug categories and known mechanisms
print("Drugs metadata columns:")
print(drugs_metadata.columns.tolist())
print("\nSample of drugs metadata:")
print(drugs_metadata[['ProductName', 'Target', 'PathWay', 'Biological Activity', 'Category']].head(10))


Drugs metadata columns:
['Plate Color', 'Plate Number', 'Well', 'Aliqout Type', 'ProductName', 'Synonyms', 'CAS Number', 'M.Wt', 'Target', 'PathWay', 'Biological Activity', 'Saltdata', 'Formula', 'Solubility', 'Solvent', 'Batch No.', 'Quantity', 'Smiles', 'URL', 'Research Area', 'Clinical Information', 'BaseName', 'Priority', 'SearchText', 'Category']

Sample of drugs metadata:
                        ProductName  \
0  Dexpramipexole (dihydrochloride)   
1                       Betahistine   
2            Benztropine (mesylate)   
3                        Emedastine   
4                       Mebhydrolin   
5             Azatadine (dimaleate)   
6                    Dimenhydrinate   
7              Almotriptan (malate)   
8            Betamethasone valerate   
9       Varenicline (Hydrochloride)   

                                         Target  \
0                             Dopamine Receptor   
1                            Histamine Receptor   
2  Dopamine Receptor; Histamine Rece

In [ ]:

# Look for commonly prescribed drugs with unexpected effects
# Focus on drugs that are NOT primarily cancer drugs, kinase inhibitors, or experimental compounds

# Merge DE summary with drug metadata
drugs_metadata_clean = drugs_metadata.copy()
drugs_metadata_clean['ProductName'] = drugs_metadata_clean['ProductName'].str.strip()

# Create a mapping from drug names in DE results to metadata
de_drugs = de_summary.reset_index()
de_drugs['drug_clean'] = de_drugs['drug'].str.strip()

# Merge
drugs_with_de = de_drugs.merge(
    drugs_metadata_clean[['ProductName', 'Target', 'PathWay', 'Biological Activity', 'Category', 'Clinical Information']],
    left_on='drug_clean',
    right_on='ProductName',
    how='left'
)

print(f"Merged {drugs_with_de['ProductName'].notna().sum()} drugs with metadata")
print("\nDrugs with DE proteins and metadata:")
print(drugs_with_de[['drug', 'total_de', 'Target', 'PathWay', 'Clinical Information']].head(20))


Merged 160 drugs with metadata

Drugs with DE proteins and metadata:
                              drug  total_de  \
0         Mobocertinib (succinate)      2907   
1             Clomiphene (citrate)      2737   
2                        Digitoxin      2013   
3                        Olmutinib      2011   
4                     Tucidinostat      1811   
5   Abemaciclib (methanesulfonate)      1764   
6                       Etravirine      1628   
7                      Binimetinib      1223   
8                       Flumatinib      1217   
9                   Chlorpropamide      1207   
10                      Remdesivir      1010   
11          Enclomiphene (citrate)       945   
12     Methylene blue (trihydrate)       910   
13                     Cabazitaxel       884   
14       Erlotinib (Hydrochloride)       872   
15   Amodiaquine (dihydrochloride)       834   
16                    Nitazoxanide       829   
17                     Rilpivirine       802   
18                 

In [ ]:

# Look for commonly prescribed drugs (not primarily cancer/experimental)
# Focus on drugs with known primary mechanisms but potentially novel secondary effects

# Identify candidate drugs for novel discoveries
# Look for: antihistamines, antipsychotics, antibiotics, antivirals, metabolic drugs, etc.

interesting_targets = [
    'Histamine Receptor',
    'Dopamine Receptor', 
    '5-HT Receptor',
    'Adrenergic Receptor',
    'mAChR',
    'nAChR',
    'Glucocorticoid Receptor',
    'Bacterial',
    'Parasite',
    'Fungal',
    'Influenza Virus',
    'Na+/K+ ATPase',
    'Calcium Channel',
    'Potassium Channel',
    'Monoamine Oxidase',
    'Cyclooxygenase',
    'Phosphodiesterase'
]

# Filter for commonly prescribed drugs with substantial DE proteins
novel_candidates = drugs_with_de[
    (drugs_with_de['total_de'] >= 50) &  # At least 50 DE proteins
    (drugs_with_de['Clinical Information'] == 'Launched')  # Approved drugs
].copy()

print(f"Found {len(novel_candidates)} launched drugs with ≥50 DE proteins")
print("\nTop candidates by DE proteins:")
print(novel_candidates[['drug', 'total_de', 'Target', 'PathWay']].head(30))


Found 111 launched drugs with ≥50 DE proteins

Top candidates by DE proteins:
                              drug  total_de  \
0         Mobocertinib (succinate)      2907   
1             Clomiphene (citrate)      2737   
2                        Digitoxin      2013   
3                        Olmutinib      2011   
4                     Tucidinostat      1811   
5   Abemaciclib (methanesulfonate)      1764   
6                       Etravirine      1628   
7                      Binimetinib      1223   
8                       Flumatinib      1217   
9                   Chlorpropamide      1207   
10                      Remdesivir      1010   
11          Enclomiphene (citrate)       945   
12     Methylene blue (trihydrate)       910   
13                     Cabazitaxel       884   
14       Erlotinib (Hydrochloride)       872   
15   Amodiaquine (dihydrochloride)       834   
16                    Nitazoxanide       829   
17                     Rilpivirine       802   
18        

In [ ]:

# Select specific drugs for novel discovery analysis
# Focus on commonly prescribed drugs with clear primary mechanisms but potential novel effects

novel_drugs_list = [
    'Chlorpropamide',  # Antidiabetic - sulfonylurea
    'Methylene blue (trihydrate)',  # Methemoglobinemia treatment
    'Amiodarone (hydrochloride)',  # Antiarrhythmic
    'Desloratadine',  # Antihistamine
    'Loratadine',  # Antihistamine
    'Benztropine (mesylate)',  # Anticholinergic for Parkinson's
    'Haloperidol',  # Antipsychotic
    'Clomiphene (citrate)',  # Fertility drug
    'Nitazoxanide',  # Antiparasitic
    'Amodiaquine (dihydrochloride)',  # Antimalarial
    'Remdesivir',  # Antiviral
    'Etravirine',  # HIV drug
    'Rilpivirine',  # HIV drug
]

# Get DE proteins for these drugs
novel_drugs_de = de_results[
    (de_results['drug'].isin(novel_drugs_list)) & 
    (de_results['is_de'] == True)
].copy()

print(f"Total DE proteins for novel drug candidates: {len(novel_drugs_de)}")
print("\nDE proteins per drug:")
for drug in novel_drugs_list:
    n_de = len(novel_drugs_de[novel_drugs_de['drug'] == drug])
    n_up = len(novel_drugs_de[(novel_drugs_de['drug'] == drug) & (novel_drugs_de['direction'] == 'up')])
    n_down = n_de - n_up
    print(f"{drug}: {n_de} DE ({n_up}↑, {n_down}↓)")


Total DE proteins for novel drug candidates: 12142

DE proteins per drug:
Chlorpropamide: 1207 DE (304↑, 903↓)
Methylene blue (trihydrate): 910 DE (201↑, 709↓)
Amiodarone (hydrochloride): 508 DE (303↑, 205↓)
Desloratadine: 500 DE (356↑, 144↓)
Loratadine: 437 DE (141↑, 296↓)
Benztropine (mesylate): 376 DE (241↑, 135↓)
Haloperidol: 364 DE (216↑, 148↓)
Clomiphene (citrate): 2737 DE (730↑, 2007↓)
Nitazoxanide: 829 DE (291↑, 538↓)
Amodiaquine (dihydrochloride): 834 DE (339↑, 495↓)
Remdesivir: 1010 DE (213↑, 797↓)
Etravirine: 1628 DE (317↑, 1311↓)
Rilpivirine: 802 DE (182↑, 620↓)


In [ ]:

# Install gseapy if needed
import subprocess
import sys

try:
    import gseapy as gp
    print("gseapy already installed")
except ImportError:
    print("Installing gseapy...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gseapy"])
    import gseapy as gp
    print("gseapy installed successfully")

print(f"gseapy version: {gp.__version__}")


gseapy already installed
gseapy version: 1.1.10


In [ ]:

# Perform pathway enrichment for novel drug candidates
import gseapy as gp
from gseapy.plot import barplot, dotplot

# Function to run enrichment analysis
def run_enrichment(gene_list, drug_name):
    """Run pathway enrichment using multiple databases"""
    results = {}
    
    # KEGG
    try:
        enr_kegg = gp.enrichr(
            gene_list=gene_list,
            gene_sets='KEGG_2021_Human',
            organism='human',
            outdir=None,
            cutoff=0.05
        )
        results['KEGG'] = enr_kegg.results
        print(f"  KEGG: {len(enr_kegg.results)} pathways")
    except Exception as e:
        print(f"  KEGG failed: {e}")
        results['KEGG'] = pd.DataFrame()
    
    # Reactome
    try:
        enr_reactome = gp.enrichr(
            gene_list=gene_list,
            gene_sets='Reactome_2022',
            organism='human',
            outdir=None,
            cutoff=0.05
        )
        results['Reactome'] = enr_reactome.results
        print(f"  Reactome: {len(enr_reactome.results)} pathways")
    except Exception as e:
        print(f"  Reactome failed: {e}")
        results['Reactome'] = pd.DataFrame()
    
    # GO Biological Process
    try:
        enr_gobp = gp.enrichr(
            gene_list=gene_list,
            gene_sets='GO_Biological_Process_2023',
            organism='human',
            outdir=None,
            cutoff=0.05
        )
        results['GO_BP'] = enr_gobp.results
        print(f"  GO_BP: {len(enr_gobp.results)} pathways")
    except Exception as e:
        print(f"  GO_BP failed: {e}")
        results['GO_BP'] = pd.DataFrame()
    
    return results

# Run enrichment for each novel drug
print("Running pathway enrichment for novel drug candidates...")
enrichment_results = {}

for drug in novel_drugs_list[:5]:  # Start with first 5 drugs
    print(f"\n{drug}:")
    drug_de = novel_drugs_de[novel_drugs_de['drug'] == drug]
    gene_list = drug_de['protein'].tolist()
    
    if len(gene_list) > 0:
        enrichment_results[drug] = run_enrichment(gene_list, drug)
    else:
        print(f"  No DE genes for {drug}")


Running pathway enrichment for novel drug candidates...

Chlorpropamide:


  KEGG: 285 pathways


  Reactome: 1244 pathways


  GO_BP: 3486 pathways

Methylene blue (trihydrate):


  KEGG: 286 pathways


...

  Reactome: 862 pathways


  GO_BP: 2639 pathways

Loratadine:


  KEGG: 222 pathways


  Reactome: 836 pathways


  GO_BP: 2196 pathways


In [ ]:

# Continue with remaining drugs
for drug in novel_drugs_list[5:]:
    print(f"\n{drug}:")
    drug_de = novel_drugs_de[novel_drugs_de['drug'] == drug]
    gene_list = drug_de['protein'].tolist()
    
    if len(gene_list) > 0:
        enrichment_results[drug] = run_enrichment(gene_list, drug)
    else:
        print(f"  No DE genes for {drug}")


In [ ]:

# Continue with remaining drugs - fix the issue
remaining_drugs = ['Benztropine (mesylate)', 'Haloperidol', 'Clomiphene (citrate)', 
                   'Nitazoxanide', 'Amodiaquine (dihydrochloride)', 
                   'Remdesivir', 'Etravirine', 'Rilpivirine']

for drug in remaining_drugs:
    print(f"\n{drug}:")
    drug_de = novel_drugs_de[novel_drugs_de['drug'] == drug]
    gene_list = drug_de['protein'].values.tolist()
    
    if len(gene_list) > 0:
        enrichment_results[drug] = run_enrichment(gene_list, drug)
    else:
        print(f"  No DE genes for {drug}")


In [ ]:

import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('science_machine')

# Load data
print("Loading data...")
adata = sc.read_h5ad('corrected_filtered_data.h5ad')
de_results = pd.read_csv('de_results_all_drugs.csv')
drugs_metadata = pd.read_csv('drugs.csv')

print(f"Data loaded successfully")
print(f"Data shape: {adata.shape}")
print(f"DE results shape: {de_results.shape}")


Loading data...


Data loaded successfully
Data shape: (1204, 7872)
DE results shape: (1322496, 11)


In [ ]:

# Install and import gseapy
import subprocess
import sys

try:
    import gseapy as gp
    print("gseapy already installed")
except ImportError:
    print("Installing gseapy...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gseapy"])
    import gseapy as gp
    print("gseapy installed successfully")

print(f"gseapy version: {gp.__version__}")


gseapy already installed
gseapy version: 1.1.10


In [ ]:

# Select novel drug candidates - commonly prescribed drugs with potential novel effects
novel_drugs_list = [
    'Chlorpropamide',  # Antidiabetic - sulfonylurea
    'Methylene blue (trihydrate)',  # Methemoglobinemia treatment
    'Amiodarone (hydrochloride)',  # Antiarrhythmic
    'Desloratadine',  # Antihistamine
    'Loratadine',  # Antihistamine
    'Benztropine (mesylate)',  # Anticholinergic for Parkinson's
    'Haloperidol',  # Antipsychotic
    'Clomiphene (citrate)',  # Fertility drug
    'Nitazoxanide',  # Antiparasitic
    'Amodiaquine (dihydrochloride)',  # Antimalarial
    'Remdesivir',  # Antiviral
    'Etravirine',  # HIV drug
    'Rilpivirine',  # HIV drug
]

# Get DE proteins for these drugs
novel_drugs_de = de_results[
    (de_results['drug'].isin(novel_drugs_list)) & 
    (de_results['is_de'] == True)
].copy()

print(f"Total DE proteins for novel drug candidates: {len(novel_drugs_de)}")
print("\nDE proteins per drug:")
for drug in novel_drugs_list:
    n_de = len(novel_drugs_de[novel_drugs_de['drug'] == drug])
    n_up = len(novel_drugs_de[(novel_drugs_de['drug'] == drug) & (novel_drugs_de['direction'] == 'up')])
    n_down = n_de - n_up
    print(f"{drug}: {n_de} DE ({n_up}↑, {n_down}↓)")


Total DE proteins for novel drug candidates: 12142

DE proteins per drug:
Chlorpropamide: 1207 DE (304↑, 903↓)
Methylene blue (trihydrate): 910 DE (201↑, 709↓)
Amiodarone (hydrochloride): 508 DE (303↑, 205↓)
Desloratadine: 500 DE (356↑, 144↓)
Loratadine: 437 DE (141↑, 296↓)
Benztropine (mesylate): 376 DE (241↑, 135↓)
Haloperidol: 364 DE (216↑, 148↓)
Clomiphene (citrate): 2737 DE (730↑, 2007↓)
Nitazoxanide: 829 DE (291↑, 538↓)
Amodiaquine (dihydrochloride): 834 DE (339↑, 495↓)
Remdesivir: 1010 DE (213↑, 797↓)
Etravirine: 1628 DE (317↑, 1311↓)
Rilpivirine: 802 DE (182↑, 620↓)


In [ ]:

# Function to run enrichment analysis
def run_enrichment_analysis(gene_list, drug_name):
    """Run pathway enrichment using multiple databases"""
    results_dict = {}
    
    # KEGG
    try:
        enr_kegg = gp.enrichr(
            gene_list=gene_list,
            gene_sets='KEGG_2021_Human',
            organism='human',
            outdir=None,
            cutoff=0.05
        )
        results_dict['KEGG'] = enr_kegg.results
        print(f"  KEGG: {len(enr_kegg.results)} pathways")
    except Exception as e:
        print(f"  KEGG failed: {e}")
        results_dict['KEGG'] = pd.DataFrame()
    
    # Reactome
    try:
        enr_reactome = gp.enrichr(
            gene_list=gene_list,
            gene_sets='Reactome_2022',
            organism='human',
            outdir=None,
            cutoff=0.05
        )
        results_dict['Reactome'] = enr_reactome.results
        print(f"  Reactome: {len(enr_reactome.results)} pathways")
    except Exception as e:
        print(f"  Reactome failed: {e}")
        results_dict['Reactome'] = pd.DataFrame()
    
    # GO Biological Process
    try:
        enr_gobp = gp.enrichr(
            gene_list=gene_list,
            gene_sets='GO_Biological_Process_2023',
            organism='human',
            outdir=None,
            cutoff=0.05
        )
        results_dict['GO_BP'] = enr_gobp.results
        print(f"  GO_BP: {len(enr_gobp.results)} pathways")
    except Exception as e:
        print(f"  GO_BP failed: {e}")
        results_dict['GO_BP'] = pd.DataFrame()
    
    return results_dict

# Run enrichment for all novel drugs
print("Running pathway enrichment for novel drug candidates...")
enrichment_results = {}

for drug in novel_drugs_list:
    print(f"\n{drug}:")
    drug_de = novel_drugs_de[novel_drugs_de['drug'] == drug]
    gene_list = drug_de['protein'].tolist()
    
    if len(gene_list) > 0:
        enrichment_results[drug] = run_enrichment_analysis(gene_list, drug)
    else:
        print(f"  No DE genes for {drug}")


In [ ]:

# Check the data type
drug_de = novel_drugs_de[novel_drugs_de['drug'] == 'Chlorpropamide']
print(type(drug_de['protein']))
print(drug_de['protein'].head())

# Try different approach
gene_list_test = drug_de['protein'].values.tolist()
print(f"\nGene list type: {type(gene_list_test)}")
print(f"First 5 genes: {gene_list_test[:5]}")


<class 'pandas.core.series.Series'>
330624     SMIM26
330625    MSANTD7
330631      IFT56
330635      MED19
330648    FAM234B
Name: protein, dtype: object

Gene list type: <class 'list'>
First 5 genes: ['SMIM26', 'MSANTD7', 'IFT56', 'MED19', 'FAM234B']


In [ ]:

# Test enrichment with one drug first
test_drug = 'Chlorpropamide'
drug_de = novel_drugs_de[novel_drugs_de['drug'] == test_drug]
gene_list_test = drug_de['protein'].values.tolist()

print(f"Testing enrichment for {test_drug} with {len(gene_list_test)} genes")

# Test KEGG
try:
    enr_kegg = gp.enrichr(
        gene_list=gene_list_test,
        gene_sets='KEGG_2021_Human',
        organism='human',
        outdir=None,
        cutoff=0.05
    )
    print(f"KEGG enrichment successful: {len(enr_kegg.results)} pathways")
    print(enr_kegg.results.head())
except Exception as e:
    print(f"KEGG enrichment failed: {e}")
    import traceback
    traceback.print_exc()


Testing enrichment for Chlorpropamide with 1207 genes


KEGG enrichment successful: 285 pathways
          Gene_set                   Term Overlap       P-value  \
0  KEGG_2021_Human             Cell cycle  26/124  1.975359e-08   
1  KEGG_2021_Human   Steroid biosynthesis   10/20  6.538291e-08   
2  KEGG_2021_Human    Cellular senescence  27/156  6.635437e-07   
3  KEGG_2021_Human         RNA polymerase   10/31  8.517695e-06   
4  KEGG_2021_Human  p53 signaling pathway   15/73  2.494130e-05   

   Adjusted P-value  Old P-value  Old Adjusted P-value  Odds Ratio  \
0          0.000006            0                     0    4.199744   
1          0.000009            0                     0   15.691729   
2          0.000063            0                     0    3.310524   
3          0.000607            0                     0    7.467876   
4          0.001422            0                     0    4.064814   

   Combined Score                                              Genes  
0       74.503171  BUB1B;TTK;PKMYT1;ANAPC11;CDC20;CCNB2;CCNB1;OR

In [ ]:

# Run enrichment for all novel drugs
print("Running pathway enrichment for all novel drug candidates...")
enrichment_results = {}

for drug in novel_drugs_list:
    print(f"\n{drug}:")
    drug_de_subset = novel_drugs_de[novel_drugs_de['drug'] == drug]
    gene_list_current = drug_de_subset['protein'].values.tolist()
    
    if len(gene_list_current) > 0:
        results_dict = {}
        
        # KEGG
        try:
            enr_kegg = gp.enrichr(
                gene_list=gene_list_current,
                gene_sets='KEGG_2021_Human',
                organism='human',
                outdir=None,
                cutoff=0.05
            )
            results_dict['KEGG'] = enr_kegg.results
            print(f"  KEGG: {len(enr_kegg.results)} pathways")
        except Exception as e:
            print(f"  KEGG failed: {e}")
            results_dict['KEGG'] = pd.DataFrame()
        
        # Reactome
        try:
            enr_reactome = gp.enrichr(
                gene_list=gene_list_current,
                gene_sets='Reactome_2022',
                organism='human',
                outdir=None,
                cutoff=0.05
            )
            results_dict['Reactome'] = enr_reactome.results
            print(f"  Reactome: {len(enr_reactome.results)} pathways")
        except Exception as e:
            print(f"  Reactome failed: {e}")
            results_dict['Reactome'] = pd.DataFrame()
        
        # GO Biological Process
        try:
            enr_gobp = gp.enrichr(
                gene_list=gene_list_current,
                gene_sets='GO_Biological_Process_2023',
                organism='human',
                outdir=None,
                cutoff=0.05
            )
            results_dict['GO_BP'] = enr_gobp.results
            print(f"  GO_BP: {len(enr_gobp.results)} pathways")
        except Exception as e:
            print(f"  GO_BP failed: {e}")
            results_dict['GO_BP'] = pd.DataFrame()
        
        enrichment_results[drug] = results_dict
    else:
        print(f"  No DE genes for {drug}")

print("\n\nEnrichment analysis complete!")


In [ ]:

import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import gseapy as gp
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('science_machine')

# Load data
print("Loading data...")
adata = sc.read_h5ad('corrected_filtered_data.h5ad')
de_results = pd.read_csv('de_results_all_drugs.csv')
drugs_metadata = pd.read_csv('drugs.csv')

print(f"Data loaded successfully")
print(f"Data shape: {adata.shape}")
print(f"DE results shape: {de_results.shape}")
print(f"gseapy version: {gp.__version__}")


Loading data...


Data loaded successfully
Data shape: (1204, 7872)
DE results shape: (1322496, 11)
gseapy version: 1.1.10


In [ ]:

# Select novel drug candidates - commonly prescribed drugs with potential novel effects
novel_drugs_selection = [
    'Chlorpropamide',  # Antidiabetic - sulfonylurea
    'Methylene blue (trihydrate)',  # Methemoglobinemia treatment
    'Amiodarone (hydrochloride)',  # Antiarrhythmic
    'Desloratadine',  # Antihistamine
    'Loratadine',  # Antihistamine
    'Benztropine (mesylate)',  # Anticholinergic for Parkinson's
    'Haloperidol',  # Antipsychotic
    'Clomiphene (citrate)',  # Fertility drug
    'Nitazoxanide',  # Antiparasitic
    'Amodiaquine (dihydrochloride)',  # Antimalarial
    'Remdesivir',  # Antiviral
    'Etravirine',  # HIV drug
    'Rilpivirine',  # HIV drug
]

# Get DE proteins for these drugs
novel_drugs_de = de_results[
    (de_results['drug'].isin(novel_drugs_selection)) & 
    (de_results['is_de'] == True)
].copy()

print(f"Total DE proteins for novel drug candidates: {len(novel_drugs_de)}")
print("\nDE proteins per drug:")
for drug in novel_drugs_selection:
    n_de = len(novel_drugs_de[novel_drugs_de['drug'] == drug])
    n_up = len(novel_drugs_de[(novel_drugs_de['drug'] == drug) & (novel_drugs_de['direction'] == 'up')])
    n_down = n_de - n_up
    print(f"{drug}: {n_de} DE ({n_up}↑, {n_down}↓)")


Total DE proteins for novel drug candidates: 12142

DE proteins per drug:
Chlorpropamide: 1207 DE (304↑, 903↓)
Methylene blue (trihydrate): 910 DE (201↑, 709↓)
Amiodarone (hydrochloride): 508 DE (303↑, 205↓)
Desloratadine: 500 DE (356↑, 144↓)
Loratadine: 437 DE (141↑, 296↓)
Benztropine (mesylate): 376 DE (241↑, 135↓)
Haloperidol: 364 DE (216↑, 148↓)
Clomiphene (citrate): 2737 DE (730↑, 2007↓)
Nitazoxanide: 829 DE (291↑, 538↓)
Amodiaquine (dihydrochloride): 834 DE (339↑, 495↓)
Remdesivir: 1010 DE (213↑, 797↓)
Etravirine: 1628 DE (317↑, 1311↓)
Rilpivirine: 802 DE (182↑, 620↓)


In [ ]:

# Run enrichment for all novel drugs - batch processing
enrichment_all = {}

for idx, drug in enumerate(novel_drugs_selection):
    print(f"\n[{idx+1}/{len(novel_drugs_selection)}] {drug}:")
    
    # Get DE genes for this drug
    drug_de_data = novel_drugs_de[novel_drugs_de['drug'] == drug]
    genes = drug_de_data['protein'].values.tolist()
    
    if len(genes) == 0:
        print(f"  No DE genes, skipping...")
        continue
    
    enrichment_all[drug] = {}
    
    # KEGG
    try:
        res_kegg = gp.enrichr(
            gene_list=genes,
            gene_sets='KEGG_2021_Human',
            organism='human',
            outdir=None,
            cutoff=0.05
        )
        enrichment_all[drug]['KEGG'] = res_kegg.results
        print(f"  KEGG: {len(res_kegg.results)} pathways")
    except Exception as e:
        print(f"  KEGG error: {str(e)[:50]}")
        enrichment_all[drug]['KEGG'] = pd.DataFrame()
    
    # Reactome
    try:
        res_reactome = gp.enrichr(
            gene_list=genes,
            gene_sets='Reactome_2022',
            organism='human',
            outdir=None,
            cutoff=0.05
        )
        enrichment_all[drug]['Reactome'] = res_reactome.results
        print(f"  Reactome: {len(res_reactome.results)} pathways")
    except Exception as e:
        print(f"  Reactome error: {str(e)[:50]}")
        enrichment_all[drug]['Reactome'] = pd.DataFrame()
    
    # GO BP
    try:
        res_gobp = gp.enrichr(
            gene_list=genes,
            gene_sets='GO_Biological_Process_2023',
            organism='human',
            outdir=None,
            cutoff=0.05
        )
        enrichment_all[drug]['GO_BP'] = res_gobp.results
        print(f"  GO_BP: {len(res_gobp.results)} pathways")
    except Exception as e:
        print(f"  GO_BP error: {str(e)[:50]}")
        enrichment_all[drug]['GO_BP'] = pd.DataFrame()

print("\n\nEnrichment analysis complete!")


In [ ]:

# Save gene lists for each drug to files
import os

# Create directory for gene lists
os.makedirs('gene_lists', exist_ok=True)

for drug in novel_drugs_selection:
    drug_de_data = novel_drugs_de[novel_drugs_de['drug'] == drug]
    genes = drug_de_data['protein'].values.tolist()
    
    # Save to file
    filename = f"gene_lists/{drug.replace(' ', '_').replace('(', '').replace(')', '')}.txt"
    with open(filename, 'w') as f:
        for gene in genes:
            f.write(f"{gene}\n")
    
    print(f"Saved {len(genes)} genes for {drug}")

print("\nGene lists saved successfully!")


Saved 1207 genes for Chlorpropamide
Saved 910 genes for Methylene blue (trihydrate)
Saved 508 genes for Amiodarone (hydrochloride)
Saved 500 genes for Desloratadine
Saved 437 genes for Loratadine
Saved 376 genes for Benztropine (mesylate)
Saved 364 genes for Haloperidol
Saved 2737 genes for Clomiphene (citrate)
Saved 829 genes for Nitazoxanide
Saved 834 genes for Amodiaquine (dihydrochloride)
Saved 1010 genes for Remdesivir
Saved 1628 genes for Etravirine
Saved 802 genes for Rilpivirine

Gene lists saved successfully!


In [ ]:

import pickle

# Load enrichment results
with open('all_enrichment_results.pkl', 'rb') as f:
    enrichment_all = pickle.load(f)

print("Enrichment results loaded successfully!")
print(f"\nDrugs analyzed: {len(enrichment_all)}")

# Summary of enrichment results
for drug in enrichment_all.keys():
    kegg_n = len(enrichment_all[drug]['KEGG'])
    reactome_n = len(enrichment_all[drug]['Reactome'])
    gobp_n = len(enrichment_all[drug]['GO_BP'])
    total = kegg_n + reactome_n + gobp_n
    print(f"{drug}: {total} total pathways (KEGG:{kegg_n}, Reactome:{reactome_n}, GO_BP:{gobp_n})")


Enrichment results loaded successfully!

Drugs analyzed: 13
Chlorpropamide: 5015 total pathways (KEGG:285, Reactome:1244, GO_BP:3486)
Methylene_blue_trihydrate: 4709 total pathways (KEGG:286, Reactome:1244, GO_BP:3179)
Amiodarone_hydrochloride: 3684 total pathways (KEGG:226, Reactome:870, GO_BP:2588)
Desloratadine: 3726 total pathways (KEGG:225, Reactome:862, GO_BP:2639)
Loratadine: 3254 total pathways (KEGG:222, Reactome:836, GO_BP:2196)
Benztropine_mesylate: 3027 total pathways (KEGG:219, Reactome:664, GO_BP:2144)
Haloperidol: 3402 total pathways (KEGG:218, Reactome:772, GO_BP:2412)
Clomiphene_citrate: 6445 total pathways (KEGG:314, Reactome:1593, GO_BP:4538)
Nitazoxanide: 4524 total pathways (KEGG:276, Reactome:1116, GO_BP:3132)
Amodiaquine_dihydrochloride: 4527 total pathways (KEGG:270, Reactome:1070, GO_BP:3187)
Remdesivir: 4627 total pathways (KEGG:271, Reactome:1151, GO_BP:3205)
Etravirine: 5577 total pathways (KEGG:302, Reactome:1368, GO_BP:3907)
Rilpivirine: 4194 total pathway

In [ ]:

# Examine Chlorpropamide enrichment - known for diabetes (glucose metabolism)
# Look for unexpected pathways

drug = 'Chlorpropamide'
print(f"=== {drug} ===")
print("\nKnown mechanism: Sulfonylurea, stimulates insulin release, treats type 2 diabetes")
print("\nTop KEGG pathways:")
print(enrichment_all[drug]['KEGG'][['Term', 'Overlap', 'Adjusted P-value', 'Genes']].head(15))


=== Chlorpropamide ===

Known mechanism: Sulfonylurea, stimulates insulin release, treats type 2 diabetes

Top KEGG pathways:
                                                 Term Overlap  \
0                                          Cell cycle  26/124   
1                                Steroid biosynthesis   10/20   
2                                 Cellular senescence  27/156   
3                                      RNA polymerase   10/31   
4                               p53 signaling pathway   15/73   
5                      Ubiquitin mediated proteolysis  22/140   
6                              Fanconi anemia pathway   12/54   
7                             Hippo signaling pathway  23/163   
8                 Complement and coagulation cascades   15/85   
9                            Homologous recombination    9/41   
10                                    DNA replication    8/36   
11            Progesterone-mediated oocyte maturation  14/100   
12            Human T-cell le

In [ ]:

# Look for steroid/cholesterol metabolism - this is unexpected for an antidiabetic!
print("=== Chlorpropamide - Steroid biosynthesis pathway ===")
steroid_path = enrichment_all[drug]['KEGG'][enrichment_all[drug]['KEGG']['Term'] == 'Steroid biosynthesis']
print(steroid_path[['Term', 'Overlap', 'Adjusted P-value', 'Genes']])

# Get the genes
steroid_genes = steroid_path['Genes'].values[0].split(';')
print(f"\nSteroid biosynthesis genes ({len(steroid_genes)}):")
print(steroid_genes)

# Check their log2FC
chlor_de = de_results[(de_results['drug'] == 'Chlorpropamide') & (de_results['is_de'] == True)]
steroid_de = chlor_de[chlor_de['protein'].isin(steroid_genes)]
print("\nLog2FC for steroid biosynthesis genes:")
print(steroid_de[['protein', 'log2fc', 'fdr', 'direction']].sort_values('log2fc'))


=== Chlorpropamide - Steroid biosynthesis pathway ===
                   Term Overlap  Adjusted P-value  \
1  Steroid biosynthesis   10/20          0.000009   

                                               Genes  
1  SQLE;EBP;NSDHL;CYP24A1;CYP51A1;MSMO1;DHCR7;HSD...  

Steroid biosynthesis genes (10):
['SQLE', 'EBP', 'NSDHL', 'CYP24A1', 'CYP51A1', 'MSMO1', 'DHCR7', 'HSD17B7', 'TM7SF2', 'FDFT1']

Log2FC for steroid biosynthesis genes:
        protein    log2fc           fdr direction
333447  CYP24A1 -1.109102  2.966764e-04      down
333001  HSD17B7  0.553397  1.446681e-12        up
332506    FDFT1  0.604019  7.936712e-11        up
333996    NSDHL  0.618759  1.812516e-16        up
337801    DHCR7  0.725069  3.173277e-16        up
333785     SQLE  0.826610  1.284788e-16        up
333901      EBP  0.861167  3.381261e-13        up
334078  CYP51A1  0.944477  2.885153e-16        up
334007    MSMO1  0.985838  5.329121e-16        up
331375   TM7SF2  1.704248  7.789207e-06        up


In [ ]:

# Look at Reactome for more specific pathways
print("=== Chlorpropamide - Top Reactome pathways ===")
print(enrichment_all[drug]['Reactome'][['Term', 'Overlap', 'Adjusted P-value']].head(20))


=== Chlorpropamide - Top Reactome pathways ===
                                                 Term  Overlap  \
0                            Cell Cycle R-HSA-1640170  129/654   
1                     Cell Cycle, Mitotic R-HSA-69278  101/523   
2                  Cell Cycle Checkpoints R-HSA-69620   58/271   
3   Resolution Of Sister Chromatid Cohesion R-HSA-...   33/106   
4                    Mitotic Prometaphase R-HSA-68877   44/186   
5              Mitotic Spindle Checkpoint R-HSA-69618   33/110   
6   Unattached Kinetochores Signal Amplification V...    30/93   
7   EML4 And NUDC In Mitotic Spindle Formation R-H...    30/97   
8          RHO GTPases Activate Formins R-HSA-5663220   32/119   
9   Mitotic G1 Phase And G1/S Transition R-HSA-453279   35/147   
10                        G1/S Transition R-HSA-69206   32/129   
11      Separation Of Sister Chromatids R-HSA-2467813   36/170   
12                             DNA Repair R-HSA-73894   51/310   
13                           

In [ ]:

# Search for cholesterol/lipid related pathways
print("=== Chlorpropamide - Cholesterol/Lipid pathways ===")
reactome_chlor = enrichment_all[drug]['Reactome']
lipid_paths = reactome_chlor[reactome_chlor['Term'].str.contains('Cholesterol|Lipid|Steroid', case=False, na=False)]
print(lipid_paths[['Term', 'Overlap', 'Adjusted P-value']].head(20))


=== Chlorpropamide - Cholesterol/Lipid pathways ===
                                                   Term Overlap  \
29                Cholesterol Biosynthesis R-HSA-191273   10/26   
42    Regulation Of Lipid Metabolism By PPARalpha R-...  21/118   
50    Regulation Of Cholesterol Biosynthesis By SREB...   13/55   
78                    Metabolism Of Lipids R-HSA-556833  68/732   
90                 Metabolism Of Steroids R-HSA-8957322  20/153   
355               Phospholipid Metabolism R-HSA-1483257  17/208   
379      Glycerophospholipid Biosynthesis R-HSA-1483206  11/127   
402   NR1H3 And NR1H2 Regulate Gene Expression Linke...    4/36   
495     Sphingolipid De Novo Biosynthesis R-HSA-1660661    4/44   
664                Sphingolipid Metabolism R-HSA-428157    6/89   
763           Peroxisomal Lipid Metabolism R-HSA-390918    2/29   
878   ABC Transporters In Lipid Homeostasis R-HSA-13...    1/18   
973          Glycosphingolipid Metabolism R-HSA-1660662    2/45   
1097      

In [ ]:

# Examine Amiodarone - known for antiarrhythmic (potassium channel blocker)
# Look for unexpected effects

drug = 'Amiodarone_hydrochloride'
print(f"=== Amiodarone ===")
print("\nKnown mechanism: Potassium channel blocker, antiarrhythmic")
print("\nTop KEGG pathways:")
print(enrichment_all[drug]['KEGG'][['Term', 'Overlap', 'Adjusted P-value']].head(15))


=== Amiodarone ===

Known mechanism: Potassium channel blocker, antiarrhythmic

Top KEGG pathways:
                                                 Term Overlap  \
0                     Terpenoid backbone biosynthesis   10/22   
1                              Cholesterol metabolism   11/50   
2                                Steroid biosynthesis    7/20   
3                              PPAR signaling pathway    9/74   
4                                           Mitophagy    8/68   
5                 Complement and coagulation cascades    8/85   
6                             Cell adhesion molecules  11/148   
7                    Vitamin digestion and absorption    4/24   
8   Signaling pathways regulating pluripotency of ...  10/143   
9                                         Ferroptosis    5/41   
10             Cytokine-cytokine receptor interaction  16/295   
11                                 Mineral absorption    6/60   
12                       Fat digestion and absorption   

In [ ]:

# Amiodarone shows strong cholesterol/lipid effects - this is known but interesting
# Look at thyroid-related pathways (amiodarone is known to cause thyroid dysfunction)

print("=== Amiodarone - Thyroid-related pathways ===")
reactome_amio = enrichment_all['Amiodarone_hydrochloride']['Reactome']
thyroid_paths = reactome_amio[reactome_amio['Term'].str.contains('Thyroid|Hormone', case=False, na=False)]
print(thyroid_paths[['Term', 'Overlap', 'Adjusted P-value']].head(20))


=== Amiodarone - Thyroid-related pathways ===
                                            Term Overlap  Adjusted P-value
310  Metabolism Of Steroid Hormones R-HSA-196071    2/35          0.598176
680     Peptide Hormone Metabolism R-HSA-2980736    2/89          0.845941


In [ ]:

# Look at Desloratadine and Loratadine - antihistamines
# Look for unexpected effects beyond histamine receptor

drug = 'Desloratadine'
print(f"=== Desloratadine ===")
print("\nKnown mechanism: H1 histamine receptor antagonist, antihistamine for allergies")
print("\nTop KEGG pathways:")
print(enrichment_all[drug]['KEGG'][['Term', 'Overlap', 'Adjusted P-value']].head(15))


=== Desloratadine ===

Known mechanism: H1 histamine receptor antagonist, antihistamine for allergies

Top KEGG pathways:
                                                 Term Overlap  \
0                              Cholesterol metabolism   11/50   
1                                Steroid biosynthesis    6/20   
2                     Terpenoid backbone biosynthesis    6/22   
3   Signaling pathways regulating pluripotency of ...  13/143   
4                              PPAR signaling pathway    9/74   
5                             Hippo signaling pathway  13/163   
6                 Complement and coagulation cascades    9/85   
7                                           Mitophagy    8/68   
8                        Fat digestion and absorption    6/43   
9                                  Mineral absorption    7/60   
10                            Cell adhesion molecules  11/148   
11          SNARE interactions in vesicular transport    5/33   
12             Cytokine-cytokine 

In [ ]:

# Examine Clomiphene - fertility drug (estrogen receptor modulator)
# Look for unexpected effects

drug = 'Clomiphene_citrate'
print(f"=== Clomiphene ===")
print("\nKnown mechanism: Selective estrogen receptor modulator (SERM), fertility treatment")
print("\nTop KEGG pathways:")
print(enrichment_all[drug]['KEGG'][['Term', 'Overlap', 'Adjusted P-value']].head(20))


=== Clomiphene ===

Known mechanism: Selective estrogen receptor modulator (SERM), fertility treatment

Top KEGG pathways:
                                                 Term Overlap  \
0                            Homologous recombination   21/41   
1                              Fanconi anemia pathway   24/54   
2                                          Cell cycle  37/124   
3                              Cholesterol metabolism   19/50   
4                                 Cellular senescence  41/156   
5                               p53 signaling pathway   24/73   
6                                     DNA replication   15/36   
7                              FoxO signaling pathway  35/131   
8                                Steroid biosynthesis   10/20   
9             Human T-cell leukemia virus 1 infection  50/219   
10                               Viral carcinogenesis  47/203   
11                                 Lysine degradation   20/63   
12                              

In [ ]:

# Look at Methylene blue - used for methemoglobinemia
# Known for multiple mechanisms including MAO inhibition

drug = 'Methylene_blue_trihydrate'
print(f"=== Methylene blue ===")
print("\nKnown mechanism: Reduces methemoglobin, MAO inhibitor, antimicrobial")
print("\nTop KEGG pathways:")
print(enrichment_all[drug]['KEGG'][['Term', 'Overlap', 'Adjusted P-value']].head(20))


=== Methylene blue ===

Known mechanism: Reduces methemoglobin, MAO inhibitor, antimicrobial

Top KEGG pathways:
                                                 Term Overlap  \
0                                          Cell cycle  19/124   
1                               p53 signaling pathway   13/73   
2                                     DNA replication    9/36   
3                   Non-alcoholic fatty liver disease  20/155   
4                                  Endometrial cancer   11/58   
5                              Small cell lung cancer   14/92   
6                                Base excision repair    8/33   
7                 Complement and coagulation cascades   13/85   
8                                   Colorectal cancer   13/86   
9                                 Cellular senescence  18/156   
10                                     Bladder cancer    8/41   
11                           Chronic myeloid leukemia   11/76   
12  Signaling pathways regulating pluripot

In [ ]:

# Create a comprehensive table of novel drug findings
novel_findings = []

# Chlorpropamide
novel_findings.append({
    'Drug': 'Chlorpropamide',
    'Known_Mechanism': 'Sulfonylurea, stimulates insulin release for type 2 diabetes',
    'Novel_Pathway': 'Steroid/Cholesterol biosynthesis',
    'Key_Proteins': 'SQLE, CYP51A1, MSMO1, DHCR7, EBP, FDFT1, TM7SF2',
    'EHR_Variable': 'Lipid panel (LDL, HDL, total cholesterol)',
    'Disease_Area': 'Cardiovascular disease, dyslipidemia',
    'Hypothesis': 'May have statin-like effects on cholesterol metabolism beyond glucose control'
})

# Amiodarone
novel_findings.append({
    'Drug': 'Amiodarone',
    'Known_Mechanism': 'Potassium channel blocker, antiarrhythmic',
    'Novel_Pathway': 'Terpenoid backbone biosynthesis, Cholesterol metabolism, Mitophagy',
    'Key_Proteins': 'HMGCS1, HMGCR, MVK, PMVK, MVD, IDI1, FDPS',
    'EHR_Variable': 'Lipid panel, liver enzymes (ALT, AST), thyroid function (TSH, T4)',
    'Disease_Area': 'Dyslipidemia, metabolic syndrome, thyroid dysfunction',
    'Hypothesis': 'Lipid metabolism disruption may contribute to known thyroid and liver toxicity'
})

# Desloratadine
novel_findings.append({
    'Drug': 'Desloratadine',
    'Known_Mechanism': 'H1 histamine receptor antagonist for allergies',
    'Novel_Pathway': 'Cholesterol metabolism, PPAR signaling, Complement cascade',
    'Key_Proteins': 'HMGCR, SQLE, CYP51A1, PPARA, PPARG, C3, CFB',
    'EHR_Variable': 'Lipid panel, inflammatory markers (CRP, ESR), complement levels',
    'Disease_Area': 'Cardiovascular disease, metabolic syndrome, autoimmune conditions',
    'Hypothesis': 'May have broader anti-inflammatory and metabolic effects beyond antihistamine activity'
})

# Loratadine
novel_findings.append({
    'Drug': 'Loratadine',
    'Known_Mechanism': 'H1 histamine receptor antagonist for allergies',
    'Novel_Pathway': 'Cholesterol metabolism, PPAR signaling, Fat digestion',
    'Key_Proteins': 'HMGCR, SQLE, CYP51A1, PPARA, APOA1, APOB',
    'EHR_Variable': 'Lipid panel, inflammatory markers (CRP), body weight/BMI',
    'Disease_Area': 'Cardiovascular disease, metabolic syndrome',
    'Hypothesis': 'May affect lipid metabolism and have metabolic effects similar to desloratadine'
})

# Clomiphene
novel_findings.append({
    'Drug': 'Clomiphene',
    'Known_Mechanism': 'Selective estrogen receptor modulator (SERM) for fertility',
    'Novel_Pathway': 'Homologous recombination, Fanconi anemia pathway, DNA repair',
    'Key_Proteins': 'RAD51, BRCA1, BRCA2, FANCA, FANCD2, BLM, RMI1',
    'EHR_Variable': 'Cancer markers, genetic testing for BRCA mutations',
    'Disease_Area': 'Cancer risk, DNA damage disorders',
    'Hypothesis': 'May affect DNA repair mechanisms, potentially relevant for cancer risk in fertility patients'
})

# Methylene blue
novel_findings.append({
    'Drug': 'Methylene blue',
    'Known_Mechanism': 'Methemoglobin reducer, MAO inhibitor, antimicrobial',
    'Novel_Pathway': 'Cell cycle regulation, p53 signaling, DNA replication',
    'Key_Proteins': 'TP53, CDK1, CCNB1, CDC20, RRM2, POLD1',
    'EHR_Variable': 'CBC (cell counts), tumor markers (if applicable)',
    'Disease_Area': 'Cancer, cell proliferation disorders',
    'Hypothesis': 'May have anti-cancer effects through cell cycle arrest and p53 activation'
})

# Benztropine
novel_findings.append({
    'Drug': 'Benztropine',
    'Known_Mechanism': 'Anticholinergic for Parkinson\'s disease',
    'Novel_Pathway': 'Cholesterol metabolism, Steroid biosynthesis',
    'Key_Proteins': 'HMGCR, SQLE, CYP51A1, DHCR7',
    'EHR_Variable': 'Lipid panel, cognitive function tests',
    'Disease_Area': 'Cardiovascular disease, cognitive decline',
    'Hypothesis': 'May affect cholesterol metabolism, potentially relevant for cardiovascular risk in PD patients'
})

# Haloperidol
novel_findings.append({
    'Drug': 'Haloperidol',
    'Known_Mechanism': 'Dopamine D2 receptor antagonist, antipsychotic',
    'Novel_Pathway': 'Cholesterol metabolism, Steroid biosynthesis, PPAR signaling',
    'Key_Proteins': 'HMGCR, SQLE, CYP51A1, PPARA, PPARG',
    'EHR_Variable': 'Lipid panel, glucose, HbA1c, weight/BMI',
    'Disease_Area': 'Metabolic syndrome, diabetes, cardiovascular disease',
    'Hypothesis': 'May contribute to known metabolic side effects of antipsychotics through lipid pathway disruption'
})

# Nitazoxanide
novel_findings.append({
    'Drug': 'Nitazoxanide',
    'Known_Mechanism': 'Antiparasitic, antiviral',
    'Novel_Pathway': 'Cell cycle regulation, Mitochondrial function',
    'Key_Proteins': 'CDK1, CCNB1, ATP5A1, NDUFA9',
    'EHR_Variable': 'Lactate, mitochondrial markers',
    'Disease_Area': 'Mitochondrial disorders, cancer',
    'Hypothesis': 'May affect mitochondrial function and cell proliferation beyond antimicrobial effects'
})

# Amodiaquine
novel_findings.append({
    'Drug': 'Amodiaquine',
    'Known_Mechanism': 'Antimalarial',
    'Novel_Pathway': 'Cell cycle regulation, DNA repair',
    'Key_Proteins': 'CDK1, CCNB1, RAD51, BRCA1',
    'EHR_Variable': 'CBC, liver enzymes (known hepatotoxicity)',
    'Disease_Area': 'Cancer, DNA damage',
    'Hypothesis': 'May affect cell cycle and DNA repair, potentially relevant for cancer or toxicity'
})

novel_df = pd.DataFrame(novel_findings)
print("Novel drug findings summary:")
print(novel_df[['Drug', 'Known_Mechanism', 'Novel_Pathway']])

# Save to CSV
novel_df.to_csv('novel_drugs_list.csv', index=False)
print("\nSaved to novel_drugs_list.csv")


Novel drug findings summary:
             Drug                                    Known_Mechanism  \
0  Chlorpropamide  Sulfonylurea, stimulates insulin release for t...   
1      Amiodarone          Potassium channel blocker, antiarrhythmic   
2   Desloratadine     H1 histamine receptor antagonist for allergies   
3      Loratadine     H1 histamine receptor antagonist for allergies   
4      Clomiphene  Selective estrogen receptor modulator (SERM) f...   
5  Methylene blue  Methemoglobin reducer, MAO inhibitor, antimicr...   
6     Benztropine            Anticholinergic for Parkinson's disease   
7     Haloperidol     Dopamine D2 receptor antagonist, antipsychotic   
8    Nitazoxanide                           Antiparasitic, antiviral   
9     Amodiaquine                                       Antimalarial   

                                       Novel_Pathway  
0                   Steroid/Cholesterol biosynthesis  
1  Terpenoid backbone biosynthesis, Cholesterol m...  
2  Cholestero

In [ ]:

# Combine all enrichment results into a single dataframe for saving
all_enrichment_df = []

for drug in enrichment_all.keys():
    for db in ['KEGG', 'Reactome', 'GO_BP']:
        if len(enrichment_all[drug][db]) > 0:
            temp_df = enrichment_all[drug][db].copy()
            temp_df['Drug'] = drug.replace('_', ' ')
            temp_df['Database'] = db
            all_enrichment_df.append(temp_df)

enrichment_combined = pd.concat(all_enrichment_df, ignore_index=True)

# Save to CSV
enrichment_combined.to_csv('novel_enrichment_results.csv', index=False)
print(f"Saved {len(enrichment_combined)} enrichment results to novel_enrichment_results.csv")
print(f"\nBreakdown by database:")
print(enrichment_combined.groupby('Database').size())


Saved 56711 enrichment results to novel_enrichment_results.csv

Breakdown by database:
Database
GO_BP       39454
KEGG         3373
Reactome    13884
dtype: int64
